# 02 - EDA: ROAD Dataset

**Input:** `data/interim/road/*.parquet` (run `python scripts/build_dataset.py --dataset road` first).

**Questions:**
1. Class distribution per capture (ambient vs attack; Fuzzy vs Spoofing; masquerade vs fabrication)
2. Frames/sec over time, with the injection window visible
3. Top CAN IDs per class
4. Inter-arrival time per class, and per target ID (masquerade attacks keep timing - harder!)
5. Sanity check of the labelling rule: attack frames should sit only inside the metadata windows

Known limitations to state in the report: no DoS class; accelerator attacks excluded; DLC always 8.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from canids.utils.config import load_config, resolve_path

cfg = load_config("data")
results = resolve_path(cfg["paths"]["results"])
interim = resolve_path(cfg["paths"]["interim"]) / "road"
files = sorted(interim.glob("*.parquet"))
assert files, f"No parquet in {interim} - run scripts/build_dataset.py first"
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
df["capture_kind"] = np.where(df["source_file"].astype(str).str.startswith("ambient"), "ambient", "attack")
df.info(memory_usage="deep")

## 0. Ingestion report (skipped captures and why)

In [ ]:
rep = pd.read_csv(results / "ingestion_report.csv")
rep[rep["dataset"] == "road"]

## 1. Class distribution per capture

In [ ]:
counts = df.groupby(["source_file", "attack_type", "label"], observed=True).size().unstack(fill_value=0)
display(counts)
px.bar(df["label"].value_counts().reset_index(), x="label", y="count", log_y=True,
       title="ROAD frames per class (log scale)")

## 2. Frames/sec over time for each attack capture (attack frames highlighted)

In [ ]:
attack_files = df.loc[df["capture_kind"] == "attack", "source_file"].unique()
for name in attack_files[:6]:  # first few; loop over all for the appendix
    g = df[df["source_file"] == name]
    sec = np.floor(g["timestamp"] - g["timestamp"].min()).astype(int)
    r = g.assign(second=sec).groupby(["second", "label"], observed=True).size().rename("frames").reset_index()
    px.area(r, x="second", y="frames", color="label", title=f"{name}: frames/sec").show()

## 3. Top CAN IDs per class

In [ ]:
top = (df.groupby("label", observed=True)["can_id"]
         .apply(lambda s: s.value_counts().head(10))
         .rename("frames").reset_index().rename(columns={"level_1": "can_id"}))
top["can_id_hex"] = top["can_id"].map(lambda v: f"0x{v:03X}")
top

## 4. Inter-arrival time: global and per target ID

In [ ]:
df = df.sort_values(["source_file", "timestamp"], kind="stable")
df["iat_ms"] = df.groupby("source_file", observed=True)["timestamp"].diff() * 1000
df["iat_id_ms"] = df.groupby(["source_file", "can_id"], observed=True)["timestamp"].diff() * 1000
display(df.groupby(["label"], observed=True)[["iat_ms", "iat_id_ms"]].describe(percentiles=[.05, .5, .95]).T)
sample = df.dropna(subset=["iat_id_ms"]).groupby("label", observed=True).sample(n=50_000, replace=True, random_state=42)
px.histogram(sample, x="iat_id_ms", color="label", nbins=200, log_y=True, barmode="overlay",
             range_x=[0, 200], title="Per-ID inter-arrival (ms) by class")

## 5. Labelling sanity check: elapsed time of attack frames per capture

In [ ]:
df["elapsed_s"] = df["timestamp"] - df.groupby("source_file", observed=True)["timestamp"].transform("min")
df[df["label"] != "Normal"].groupby("source_file", observed=True)["elapsed_s"].agg(["min", "max", "count"])
# Compare min/max with injection_interval in capture_metadata.json

## Findings (fill in from the real outputs above - no invented numbers)
- Class balance: [TBD]
- Masquerade vs fabrication timing differences: [TBD]
- Labelling sanity check passed? [TBD]
- Differences vs HCRL that affect cross-dataset tests (IDs, DLC always 8, rates): [TBD]